# 02 — Run a Benchmark Matrix

This notebook mirrors `pipelines/run_benchmark.py`. It runs a full **tasks × frameworks × models** matrix, collects scored results, and lets you analyse them with pandas.

### What you will learn
- How `BenchmarkConfig` drives a run matrix from a YAML file
- How `BenchmarkRunner` loops over all combinations and scores each one
- How to load and analyse JSONL results with pandas

### Prerequisites
```bash
uv sync --extra biomni --extra dev
ollama pull llama3
```

In [ ]:
import sys
from pathlib import Path

import pandas as pd

repo_root = Path(".").resolve().parent
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

print(f"Repo root: {repo_root}")

## 2. Load a config

`BenchmarkConfig.from_yaml()` reads a YAML file and separates known fields (`tasks`, `frameworks`, `models`, `eval`, `output_dir`) from framework-specific kwargs (e.g. `num_queries` for Robin, `timeout_seconds` for Biomni).

In [ ]:
from bio_agents.evaluation.runner import BenchmarkConfig

# Pick a config — swap to robin_smoke.yaml to run Robin instead
CONFIG_PATH = repo_root / "configs" / "biomni_smoke.yaml"

cfg = BenchmarkConfig.from_yaml(CONFIG_PATH)

print(f"Config: {CONFIG_PATH.name}")
print(f"  tasks      : {cfg.tasks}")
print(f"  frameworks : {cfg.frameworks}")
print(f"  models     : {cfg.models}")
print(f"  output_dir : {cfg.output_dir}")
print(f"  runner_kwargs: {cfg.runner_kwargs}")

## 3. Preview the matrix (dry-run)

Before spending time on actual agent runs, preview the full matrix as a DataFrame.

In [ ]:
matrix = [
    {"task": t, "framework": f, "model": m}
    for t in cfg.tasks
    for f in cfg.frameworks
    for m in cfg.models
]

df_matrix = pd.DataFrame(matrix)
print(f"Total runs: {len(df_matrix)}")
df_matrix

## 4. Run the benchmark

`BenchmarkRunner.run()` executes every cell in the matrix and returns a list of `RunResult` objects. Each contains the agent output, tool calls, score, and timing.

In [ ]:
import time

from bio_agents.evaluation.runner import BenchmarkRunner

runner = BenchmarkRunner(cfg)

print(f"Starting {len(df_matrix)} run(s)...")
t0 = time.perf_counter()
results = runner.run()
elapsed = time.perf_counter() - t0

print(f"Done in {elapsed:.1f}s  —  {len(results)} result(s) collected")

## 5. Analyse results

In [ ]:
# Convert to a flat DataFrame for easy analysis
df = pd.DataFrame([r.to_dict() for r in results])

# Quick summary
print(f"Pass rate: {df['passed'].mean():.1%}  ({df['passed'].sum()}/{len(df)})")
print(f"Avg score: {df['score'].mean():.3f}")
print(f"Avg duration: {df['duration_s'].mean():.1f}s")
df[["task", "framework", "model", "score", "passed", "duration_s"]]

In [ ]:
# Score breakdown by task
df.groupby("task")[["score", "passed"]].agg({"score": "mean", "passed": "sum"}).rename(
    columns={"score": "avg_score", "passed": "n_passed"}
)

In [ ]:
r = results[0]
print(
    f"Task: {r.task}  |  Score: {r.eval_result.score:.2f}"
    f"  |  Passed: {r.eval_result.passed}"
)
print("\n--- Output (first 1000 chars) ---")
print((r.agent_result.output or "")[:1000])
print("\n--- Eval metrics ---")
print(r.eval_result.metrics)

## 6. Save results to JSONL

In [ ]:
out_file = runner.save(results)
print(f"Saved to: {out_file}")

# Verify by reloading
df_saved = pd.read_json(out_file, lines=True)
print(f"Rows in file: {len(df_saved)}")
df_saved.head()

## 7. Load and compare previous runs

Results accumulate in the JSONL file across runs. Use this cell to load and compare across frameworks or models.

In [ ]:
results_file = repo_root / cfg.output_dir / "results.jsonl"

if results_file.exists():
    df_all = pd.read_json(results_file, lines=True)
    print(f"Total rows across all runs: {len(df_all)}")
    df_all.groupby(["framework", "model"])["score"].mean().rename("avg_score")
else:
    print("No results file yet — run cell 4 first.")